# Data Visualization Dashboard
This notebook loads the customer analytics dataset and financial analysis output, then creates interactive visualizations for business insights.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

root = Path.cwd()
for _ in range(6):
    if (root / 'data' / 'SampleSuperstore.csv').exists() and (root / 'report' / 'financial_analysis_report.xlsx').exists():
        break
    root = root.parent

customer_path = root / 'data' / 'SampleSuperstore.csv'
financial_report_path = root / 'report' / 'financial_analysis_report.xlsx'

customer_df = pd.read_csv(customer_path, encoding='latin-1')
financial_df = pd.read_excel(financial_report_path, sheet_name='AAPL Analysis', index_col=0)

customer_df['Order Date'] = pd.to_datetime(customer_df['Order Date'])
customer_df['YearMonth'] = customer_df['Order Date'].dt.to_period('M').astype(str)
sales_trend = customer_df.groupby('YearMonth')['Sales'].sum().reset_index()
category_perf = customer_df.groupby('Category')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)

display(customer_df.head())
display(financial_df.head())


In [ ]:
fig_sales = px.line(sales_trend, x='YearMonth', y='Sales', markers=True, title='Monthly Sales Trend')
fig_sales.update_layout(xaxis_title='Month', yaxis_title='Sales', xaxis_tickangle=-45)
fig_sales.show()


In [ ]:
fig_category = px.bar(category_perf, x='Category', y='Sales', title='Sales by Category', text='Sales')
fig_category.update_layout(yaxis_title='Sales', xaxis_title='Category')
fig_category.show()


In [ ]:
fig_stock = go.Figure()
fig_stock.add_trace(go.Scatter(x=financial_df.index, y=financial_df['Close'], mode='lines', name='AAPL Close'))
fig_stock.add_trace(go.Scatter(x=financial_df.index, y=financial_df['MA_50'], mode='lines', name='50-Day MA', line=dict(dash='dash')))
fig_stock.add_trace(go.Scatter(x=financial_df.index, y=financial_df['MA_200'], mode='lines', name='200-Day MA', line=dict(dash='dot')))
fig_stock.update_layout(title='AAPL Close Price with Moving Averages', xaxis_title='Date', yaxis_title='Price ($)')
fig_stock.show()


In [ ]:
financial_df['Daily Return'] = financial_df['Daily Return'] * 100
fig_returns = px.area(financial_df, x=financial_df.index, y='Daily Return', title='AAPL Daily Return (%)', labels={'y':'Daily Return (%)'})
fig_returns.show()
